# Optimize Regex Patterns

As regex patterns grow more complex, readability and performance matter. This module covers greedy vs non-greedy matching, pattern simplification, non-capturing groups, `re.compile`, `re.DEBUG`, and common optimization strategies.

In [1]:
import re

## Greedy vs Non-Greedy Matching

By default, quantifiers are **greedy** — they match as much text as possible.
Adding `?` makes them **non-greedy** (lazy) — they match as little as possible.

| Greedy | Non-greedy | Meaning |
|---|---|---|
| `*` | `*?` | 0 or more |
| `+` | `+?` | 1 or more |
| `{n,m}` | `{n,m}?` | between n and m |

In [2]:
html = "<p>Hello</p><p>Pluralsight</p>"

# Greedy — matches the longest possible string
greedy = re.findall(r"<.*>", html)
print("Greedy:", greedy)
# ['<p>Hello</p><p>Pluralsight</p>']  — one big match

# Non-greedy — matches the shortest possible string
non_greedy = re.findall(r"<.*?>", html)
print("Non-greedy:", non_greedy)
# ['<p>', '</p>', '<p>', '</p>']  — each tag separately

Greedy: ['<p>Hello</p><p>Pluralsight</p>']
Non-greedy: ['<p>', '</p>', '<p>', '</p>']


## Simplifying Complex Patterns

A readable pattern is easier to maintain and less likely to contain bugs.

**Goal:** match hex color codes (`#fff` or `#ffffff`)

In [3]:
# Verbose — works but hard to read
pattern_verbose = r"#([a-fA-F0-9]{6}|[a-fA-F0-9]{3})"

# Simplified with non-capturing group — same result, cleaner
# (?:...) = non-capturing group
# {3}{1,2} means: a group of 3 hex chars, repeated 1 or 2 times
pattern_simple = r"#(?:[a-fA-F0-9]{3}){1,2}"

colors = ["#fff", "#ffffff", "#FFF", "#AABBCC", "#GGG"]
for c in colors:
    m1 = re.fullmatch(pattern_verbose, c)
    m2 = re.fullmatch(pattern_simple, c)
    print(f"{c}: verbose={bool(m1)}, simple={bool(m2)}")

#fff: verbose=True, simple=True
#ffffff: verbose=True, simple=True
#FFF: verbose=True, simple=True
#AABBCC: verbose=True, simple=True
#GGG: verbose=False, simple=False


## Non-Capturing Groups `(?:...)`

Use `(?:...)` when you need grouping for quantifiers or alternation but **don't** need to capture the result. This is faster than capturing groups.

In [4]:
# Capturing group — result includes the group's content
print(re.findall(r"(\d{3})-\d{4}", "555-1234 and 666-5678"))  # ['555', '666']

# Non-capturing group — result is the whole match
print(re.findall(r"(?:\d{3})-\d{4}", "555-1234 and 666-5678"))  # ['555-1234', '666-5678']

['555', '666']
['555-1234', '666-5678']


## Compiling Patterns with `re.compile`

Compile a pattern once to reuse it many times without re-parsing overhead.

In [5]:
import time

lines = ["user@example.com", "bad-email", "another@test.org"] * 10_000

email_re = re.compile(r"\b\w+@\w+\.\w+\b")

start = time.perf_counter()
results = [email_re.search(line) for line in lines]
elapsed = time.perf_counter() - start

hits = sum(1 for r in results if r)
print(f"{hits} matches in {elapsed:.4f}s (compiled pattern)")

20000 matches in 0.0084s (compiled pattern)


## Using `re.DEBUG` to Inspect Compilation

`re.DEBUG` prints how Python parsed and compiled your pattern — useful for understanding or debugging complex regex.

In [6]:
# Inspect how Python sees a simple email pattern
import io, sys

# Redirect stdout to capture debug output
old_stdout = sys.stdout
sys.stdout = buffer = io.StringIO()

try:
    re.compile(r"\b\w+@\w+\.\w+\b", re.DEBUG)
finally:
    sys.stdout = old_stdout

debug_output = buffer.getvalue()
print(debug_output)

AT AT_BOUNDARY
MAX_REPEAT 1 MAXREPEAT
  IN
    CATEGORY CATEGORY_WORD
LITERAL 64
MAX_REPEAT 1 MAXREPEAT
  IN
    CATEGORY CATEGORY_WORD
LITERAL 46
MAX_REPEAT 1 MAXREPEAT
  IN
    CATEGORY CATEGORY_WORD
AT AT_BOUNDARY

 0. INFO 4 0b0 5 MAXREPEAT (to 5)
 5: AT UNI_BOUNDARY
 7. REPEAT_ONE 9 1 MAXREPEAT (to 17)
11.   IN 4 (to 16)
13.     CATEGORY UNI_WORD
15.     FAILURE
16:   SUCCESS
17: LITERAL 0x40 ('@')
19. REPEAT_ONE 9 1 MAXREPEAT (to 29)
23.   IN 4 (to 28)
25.     CATEGORY UNI_WORD
27.     FAILURE
28:   SUCCESS
29: LITERAL 0x2e ('.')
31. REPEAT_ONE 9 1 MAXREPEAT (to 41)
35.   IN 4 (to 40)
37.     CATEGORY UNI_WORD
39.     FAILURE
40:   SUCCESS
41: AT UNI_BOUNDARY
43. SUCCESS



## Pattern Optimization Tips

1. **Be specific** — `[0-9]{4}` is faster than `\d{4}` in some engines, and avoids surprises with Unicode digits
2. **Anchor patterns** — `^` and `$` stop the engine scanning the whole string
3. **Avoid catastrophic backtracking** — patterns like `(a+)+` can cause exponential slowdown
4. **Use non-capturing groups** when you don't need the captured text
5. **Compile patterns** that are used in loops
6. **Prefer `re.fullmatch`** for validation over anchoring manually with `^...$`

In [7]:
# Catastrophic backtracking example (don't run on very long strings!)
# Pattern like (a+)+ can be exponentially slow
# Instead, flatten the quantifiers:

# Bad  pattern: r"(a+)+b"
# Better pattern: r"a+b"

print(re.search(r"a+b", "aaab"))   # fine
print(re.search(r"a+b", "aaaa"))   # no match — stops quickly

<re.Match object; span=(0, 4), match='aaab'>
None


## Verbose Mode `re.VERBOSE`

Use `re.VERBOSE` (or `re.X`) to write patterns across multiple lines with comments — the ultimate readability boost for complex patterns.

In [8]:
EMAIL_PATTERN = re.compile(r"""
    ^                       # start of string
    [a-zA-Z0-9._%+-]+       # local part
    @                       # at sign
    [a-zA-Z0-9.-]+          # domain name
    \.                      # literal dot
    [a-zA-Z]{2,}            # top-level domain
    $                       # end of string
""", re.VERBOSE)

tests = ["user@example.com", "bad@", "good@domain.org"]
for t in tests:
    print(f"{t}: {bool(EMAIL_PATTERN.match(t))}")

user@example.com: True
bad@: False
good@domain.org: True


## Putting It All Together: Optimized Log Parser

In [9]:
LOG_RE = re.compile(r"""
    ^                                       # start of line
    (?P<date>\d{4}-\d{2}-\d{2})            # date YYYY-MM-DD
    \s
    (?P<time>\d{2}:\d{2}:\d{2})            # time HH:MM:SS
    \s
    (?P<level>DEBUG|INFO|WARNING|ERROR)     # log level
    \s+
    (?P<ip>[\d.]+)                          # IP address
    \s+-\s+
    (?P<message>.+)                         # message
    $                                       # end of line
""", re.VERBOSE)

log_lines = [
    "2024-01-15 08:23:11 INFO  192.168.1.10 - Request processed",
    "2024-01-15 08:24:02 ERROR 172.16.0.8 - Connection refused [ERR-503]",
]

for line in log_lines:
    m = LOG_RE.match(line)
    if m:
        print(m.groupdict())

{'date': '2024-01-15', 'time': '08:23:11', 'level': 'INFO', 'ip': '192.168.1.10', 'message': 'Request processed'}
{'date': '2024-01-15', 'time': '08:24:02', 'level': 'ERROR', 'ip': '172.16.0.8', 'message': 'Connection refused [ERR-503]'}
